# 05 - Stage 2: audit of the stage-1 candidates (final ranking)

`Qwen/Qwen3.8-27B` (thinking, lowest effort, up to 32,768 new tokens) audits the stage-1 candidates and returns the ranked top-3 codes (`ranked_codes`). Per row the prompt contains: the candidates and the codes stage 1 considered but did not list; rule cards for the candidates and their usual confusions; annotated examples and review precedents retrieved from the knowledge base (BM25, excluding the row itself); the case; the transcriptions and review notes; the student photo and the question diagram. Prompt: `resources/prompts/stage2_audit_template_v6.txt` over the stage-2 base prompt.

This notebook is the local-inference counterpart of the notebook that produced the final submission (`05_stage2_audit_test_full`): same retrieval, same prompt construction, same parsing. Rows without an ok stage-1 record are not requested; the submission builder falls back to stage 1 for them.

**Reads** `work/stage1`, `work/pass3`, `resources/corpus/*` and `resources/reference/*` (private artifacts).  **Writes** `work/stage2/{records/,results.csv}`.


**Default: smoke test on 5 rows.** Set `LIMIT = None` in the second cell for the full run.

In [ ]:
import os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'models.json').is_file())
os.chdir(REPO); sys.path.insert(0, str(REPO / 'src'))
from misc_pipeline import common, runner
LIMIT = 5         # SMOKE TEST: first 5 test rows only. Set LIMIT = None for the full 3,498 rows (finished rows are kept and skipped, so the full run resumes from the smoke test).
print('repo:', REPO, '| data:', common.DATA, '| work:', common.WORK)

In [ ]:
from misc_pipeline import stage2, stage_exports
jobs, metas = stage2.build_jobs(LIMIT)
print(len(jobs), 'prompts built | median prompt chars:', sorted(len(j['prompt']) for j in jobs)[len(jobs) // 2])
j0 = jobs[0]['prompt']; print(j0[j0.index('THE CASE'):][:1500])

In [ ]:
summary = runner.run_stage('stage2', jobs, common.STAGE_DIRS['stage2'], stage2.parse_record, label_images=True)
print(summary)

In [ ]:
n_total = len(common.test_rows()) if LIMIT is None else LIMIT
exp = stage_exports.export_stage2(jobs, metas, n_total); print(exp)

## Free the GPUs
The stage already closes its model when it ends. Run this cell after an interrupted run or an error, or before starting the next notebook in the same kernel.

In [ ]:
import gc
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
        print('GPU memory in use (GB):', [round(torch.cuda.memory_allocated(i) / 1e9, 1) for i in range(torch.cuda.device_count())])
except ImportError:
    pass
# If memory is still held, restart the kernel (Kernel > Restart); vLLM worker processes then exit.

## Optional: delete the downloaded model
The weights stay in the Hugging Face cache (`HF_HOME`) and are re-downloaded on the next use. Set `DELETE_MODEL = True` to remove `Qwen/Qwen3.8-27B` from the cache now. Keep it if a later notebook uses the same model (notebooks 02 and 03 share one model, as do 04 and 05).

In [ ]:
DELETE_MODEL = False
MODEL = 'Qwen/Qwen3.8-27B'
if DELETE_MODEL:
    from huggingface_hub import scan_cache_dir
    info = scan_cache_dir()
    hashes = [r.commit_hash for repo in info.repos if repo.repo_id == MODEL for r in repo.revisions]
    if hashes:
        strategy = info.delete_revisions(*hashes); print('freeing %.1f GB' % (strategy.expected_freed_size / 1e9)); strategy.execute()
    else:
        print(MODEL, 'is not in the cache')
else:
    print('model kept in the cache')